In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from src.preprocessing import preprocess
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bhara\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
df = pd.read_csv('../data/fiqa.csv')
print(df.shape)
df.head()

(14511, 2)


,question,answer
0,What do brokers do with bad stock?,"For every seller, there's a buyer. Buyers may ..."
1,Why do investors buy stock that had appreciated?,"You seem to prefer to trade like I do: ""Buy lo..."
2,Is it ever a good idea to close credit cards?,"Yes, it can be a good idea to close unused cre..."
3,If something is coming into my account will it...,The bank will make this even more confusing be...
4,What one bit of financial advice do you wish y...,When I was contracting I wish I had joined a t...


In [3]:
df['tokens'] = df['question'].apply(preprocess)
print(df[['question', 'tokens']].head())

                                            question  \
0                 What do brokers do with bad stock?   
1   Why do investors buy stock that had appreciated?   
2      Is it ever a good idea to close credit cards?   
3  If something is coming into my account will it...   
4  What one bit of financial advice do you wish y...   

                                              tokens  
0                               [broker, bad, stock]  
1                [investor, buy, stock, appreciated]  
2            [ever, good, idea, close, credit, card]  
3  [something, coming, account, debit, credit, ac...  
4  [one, bit, financial, advice, wish, could, 've...  


In [4]:
model = Word2Vec(
    sentences=df['tokens'].tolist(),
    vector_size=100,
    window=5,
    min_count=1,
    epochs=10
)
print("Vocabulary size:", len(model.wv))

Vocabulary size: 5465


In [5]:
def get_avg_vector(tokens, model):
    vectors = []
    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    if len(vectors) == 0:
        return np.zeros(100)
    return np.mean(vectors, axis=0)

In [6]:
df['vector'] = df['tokens'].apply(lambda x: get_avg_vector(x, model))
question_vectors = np.vstack(df['vector'].values)
print("Question vectors shape:", question_vectors.shape)

Question vectors shape: (14511, 100)


In [7]:
def find_answer(query, top_k=3):
    tokens = preprocess(query)
    query_vector = get_avg_vector(tokens, model).reshape(1, -1)
    similarities = cosine_similarity(query_vector, question_vectors)
    top_indices = similarities[0].argsort()[-top_k:][::-1]
    
    print(f"Query: {query}\n")
    for i, idx in enumerate(top_indices):
        print(f"Result {i+1}:")
        print(f"Question: {df['question'][idx]}")
        print(f"Answer: {df['answer'][idx]}")
        print(f"Similarity Score: {similarities[0][idx]:.4f}")
        print("---")

In [8]:
find_answer("How should I invest my money in stocks?")

Query: How should I invest my money in stocks?

Result 1:
Question: Do I make money in the stock market from other people losing money?
Answer: The answer is partly and sometimes, but you cannot know when or how. Most clearly, you do not take somebody else's money if you buy shares in a start-up company. You are putting your money at risk in exchange for a share in the rewards. Later, if the company thrives, you can sell your shares for whatever somebody else will pay for your current share in the thriving company's earnings. Or, you lose your money, when the company fails. (Much of it has then ended up in the company's employees' pockets, much of the rest with the government as taxes that the company paid).  If the stockmarket did not exist, people would be far less willing to put their money into a new company, because selling shares would be far harder. This in turn would mean that fewer new things were tried out, and less progress would be made. Communists insist that central state